In [31]:
import sys
sys.path.append("../src")

from preprocessing import (
    clean_train_pipeline,
    get_feature_columns,
)

from training import(
    split_by_engine,
    prepare_train_val_data
)

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

In [32]:
train = clean_train_pipeline(
    "../data/train_FD001.txt"
)
train.shape

(20631, 27)

In [33]:
train_data, val_data = split_by_engine(train)

print("Train:", train_data.shape)
print("Validation:", val_data.shape)

Train: (16561, 27)
Validation: (4070, 27)


In [34]:
X_train, y_train, X_val, y_val = prepare_train_val_data(train_data, val_data, feature_cols=get_feature_columns(train_data))
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (16561, 18)
y_train: (16561,)
X_val: (4070, 18)
y_val: (4070,)


In [35]:
param_combinations = [
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 4},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 8},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 4},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 6},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 8},
]

In [36]:
results = []

for params in param_combinations:

    model = XGBRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)

    results.append({
        "n_estimators": params["n_estimators"],
        "learning_rate": params["learning_rate"],
        "max_depth": params["max_depth"],
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

In [ ]:
tuning_results = pd.DataFrame(results)

tuning_results.sort_values(
    "MAE",
    ascending=True
)

,n_estimators,learning_rate,max_depth,MAE,RMSE,R2
0,200,0.05,4,23.064381,30.201897,0.788372
3,300,0.05,4,23.232479,30.487684,0.784348
1,200,0.05,6,23.573032,31.188009,0.774327
4,300,0.05,6,23.757696,31.471761,0.770201
2,200,0.05,8,24.142551,31.882340,0.764166
5,300,0.05,8,24.401598,32.229514,0.759002


In [38]:
learning_rates = [0.03, 0.05, 0.10]

results_lr = []

for lr in learning_rates:

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=lr,
        max_depth=4,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)

    results_lr.append({
        "learning_rate": lr,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

learning_rate_results = pd.DataFrame(results_lr)

learning_rate_results.sort_values(
    "MAE",
    ascending=True
)

,learning_rate,MAE,RMSE,R2
0,0.03,22.833530,29.886466,0.792769
1,0.05,23.064381,30.201897,0.788372
2,0.10,23.428501,30.861322,0.779029


In [39]:
n_estimators_values = [200, 300, 400]

results_n_estimators = []

for n in n_estimators_values:

    model = XGBRegressor(
        n_estimators=n,
        learning_rate=0.03,
        max_depth=4,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)

    results_n_estimators.append({
        "n_estimators": n,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

n_estimators_results = pd.DataFrame(results_n_estimators)

n_estimators_results.sort_values(
    "MAE",
    ascending=True
)

,n_estimators,MAE,RMSE,R2
0,200,22.833530,29.886466,0.792769
1,300,22.939167,30.084260,0.790017
2,400,23.076298,30.303021,0.786952


In [40]:
min_child_weights = [1, 3, 5]

results_min_child = []

for weight in min_child_weights:

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=weight,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)

    results_min_child.append({
        "min_child_weight": weight,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

min_child_results = pd.DataFrame(results_min_child)

min_child_results.sort_values(
    "MAE",
    ascending=True
)

,min_child_weight,MAE,RMSE,R2
1,3,22.794096,29.834766,0.793486
0,1,22.833530,29.886466,0.792769
2,5,22.834888,29.882888,0.792819


In [41]:
subsample_values = [0.8, 1.0]

results_subsample = []

for subsample in subsample_values:

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=3,
        subsample=subsample,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)

    results_subsample.append({
        "subsample": subsample,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

subsample_results = pd.DataFrame(results_subsample)

subsample_results.sort_values(
    "MAE",
    ascending=True
)

,subsample,MAE,RMSE,R2
1,1.0,22.794096,29.834766,0.793486
0,0.8,22.825411,29.884027,0.792803


In [42]:
colsample_values = [0.8, 1.0]

results_colsample = []

for colsample in colsample_values:

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=3,
        colsample_bytree=colsample,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)

    results_colsample.append({
        "colsample_bytree": colsample,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

colsample_results = pd.DataFrame(results_colsample)

colsample_results.sort_values(
    "MAE",
    ascending=True
)

,colsample_bytree,MAE,RMSE,R2
0,0.8,22.749077,29.824614,0.793626
1,1.0,22.794096,29.834766,0.793486


In [43]:
"""
learning_rate      → تأثير واضح
n_estimators       → 200 كان مناسبًا في تجاربنا
min_child_weight   → تأثير بسيط
subsample          → لم يحسن النتيجة
colsample_bytree   → تحسن بسيط

"""

'\nlearning_rate      → تأثير واضح\nn_estimators       → 200 كان مناسبًا في تجاربنا\nmin_child_weight   → تأثير بسيط\nsubsample          → لم يحسن النتيجة\ncolsample_bytree   → تحسن بسيط\n\n'